---
title: Median realized volatility (MedRV)
execute:
  enabled: true
---

`q.indicator.med_rv` estimates continuous variation from the median absolute return in each overlapping three-return block. This construction is less sensitive to isolated jumps than realized variance.

The scalar estimator is applied to trailing 21-session daily log-return windows for AAPL and SPY over five years.

In [1]:
import pandas as pd

import qrt as q

prices = pd.concat(
    {
        "AAPL": q.data.datasets.load("aapl")["close"],
        "SPY": q.data.datasets.load("spy")["close"],
    },
    axis=1,
    join="inner",
).dropna()
end = prices.index.max()
prices = prices.loc[end - pd.DateOffset(years=5) :]
returns = pd.DataFrame({symbol: pd.Series(q.indicator.log_returns(prices[symbol]), index=prices.index[1:]) for symbol in prices})
returns.tail()

,AAPL,SPY
datetime,,
2026-07-14,-0.007751,0.003544
2026-07-15,0.039360,0.003956
2026-07-16,0.017435,-0.005433
2026-07-17,0.001439,-0.009946
2026-07-20,-0.021657,-0.001616


## Calculate the indicator

At least three returns are required to form the first overlapping triplet.

In [2]:
sample = returns["AAPL"].iloc[-21:]
q.indicator.med_rv(sample)

0.00866285098617099

## Compare with SPY

Rolling application creates a time series for each asset. The 10,000 multiplier keeps the variation scale readable.

In [3]:
window = 21
comparison = pd.DataFrame(
    {
        symbol: returns[symbol].rolling(window).apply(q.indicator.med_rv, raw=True)
        for symbol in returns
    }
).mul(10_000)
comparison.columns = [f"{symbol} MedRV" for symbol in comparison]
fig = q.plot.col(
    comparison,
    title=f"AAPL and SPY trailing {window}-session median realized volatility",
    ylabel="MedRV (×10,000)",
)
fig.show(renderer="notebook_connected")